In [0]:
from pyspark.sql.functions import trim, col, regexp_replace
from pyspark.sql.types import DoubleType

In [0]:
df_products = spark.table("retailer.bronze.products_raw")

display(df_products)

In [0]:
df_products_clean = (
    df_products
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("brand", trim(col("brand")))
    .withColumn("color", trim(col("color")))
    .withColumn("subcategory", trim(col("subcategory")))
    .withColumn("category", trim(col("category")))
)

In [0]:
df_products_clean = (
    df_products_clean
    .withColumn(
        "unit_cost_usd",
        regexp_replace(col("unit_cost_usd"), "[\\$,\\s]", "").cast(DoubleType())
    )
    .withColumn(
        "unit_price_usd",
        regexp_replace(col("unit_price_usd"), "[\\$,\\s]", "").cast(DoubleType())
    )
)

In [0]:
df_products_clean = (
    df_products_clean
    .filter(col("product_key").isNotNull())
    .dropDuplicates(["product_key"])
)

In [0]:
display(df_products_clean)

df_products_clean.printSchema()

In [0]:
df_products_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("retailer.silver.products")

